In [19]:
import redis
import json
import redis.commands.json as redis_json

REDIS_HOST = 'localhost'
REDIS_PASSWORD = "123456"
REDIS_PORT = 6379
INDEX_NAME = "GameObjectsIdx"
KEY_PREFIX = "GameObjects:"

redis_client = redis.StrictRedis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    # ssl=USE_SSL,
    decode_responses=True 
)

all_data = {}
# Use scan_iter for safe iteration over all keys
for key in redis_client.scan_iter('*'):
    # Determine the type of the key to use the correct retrieval command
    key_type = redis_client.type(key)
    
    try:
        if key_type == 'string':
            # For plain strings (serialized JSON), retrieve the string and parse it
            value = redis_client.get(key)
            all_data[key] = json.loads(value)
        elif key_type == 'ReJSON-RL': # This is the type returned for RedisJSON keys
            # For RedisJSON types, use the JSON.GET command
            value = redis_client.json().get(key)
            all_data[key] = value
        else:
            # Handle other types if necessary (e.g., hash, list, set)
            all_data[key] = f"(Skipped: Non-JSON type '{key_type}')"
    except json.JSONDecodeError:
        # Handle cases where a string key contains non-JSON data
        all_data[key] = f"(Skipped: Invalid JSON in string key)"
    except Exception as e:
        all_data[key] = f"(Error retrieving data: {e})"
        
for key, data in all_data.items():
    print(f"Key: **{key}**")
    print(f"Value: {data}\n")

In [20]:
redis_client.delete("GameObjects:array")

0

In [21]:
request = [{"GameObjects": [
        {
            "Id": "123456789",
            "Tag": "cube",
            "Name": "Cube_2",
            "Components": {
                "ConstantForce": 9.82,
                "Color": "red"
            },
            "Transform": {
                "Position": {
                    "X": 1.98,
                    "Y": 1.287,
                    "Z": 1.331
                },
                "Rotation": {
                    "X": 0,
                    "Y": 0,
                    "Z": 5.17
                },
                "Scale": {
                    "X": 0.04014344,
                    "Y": 0.4354826,
                    "Z": 0.03871146
                },
            }
        }
    ]}]

redis_client.json().set(f"GameObjects:array" , '$', request)

True

In [25]:
#query = "@Tag:{cube} @ComponentsColor:{red} @ComponentsConstantForce:[-inf 9.82]"
query = "@Tag:{cube}"
search_results = redis_client.ft(INDEX_NAME).search(query)
search_results

Result{0 total, docs: []}

In [24]:
try:
    redis_client.execute_command('FT.DROPINDEX', INDEX_NAME)
    print(f"Index '{INDEX_NAME}' has been deleted successfully.")
except redis.exceptions.ResponseError as e:
    print(f"Index '{INDEX_NAME}' not found. Creating index now...")

    create_command_args = [
                'FT.CREATE', INDEX_NAME,
                'ON', 'JSON',
                'PREFIX', '1', f"{KEY_PREFIX}",
                'SCHEMA',
                '$.GameObjects[*].Id', 'AS', 'Id', 'TAG',
                '$.GameObjects[*].Tag', 'AS', 'Tag', 'TAG',
                '$.GameObjects[*].Name', 'AS', 'Name', 'TAG',
                '$.GameObjects[*].Components.ConstantForce', 'AS', 'ComponentsConstantForce', 'NUMERIC',
                '$.GameObjects[*].Components.Color', 'AS', 'ComponentsColor', 'TAG',
                '$.GameObjects[*].Transform.Position.X', 'AS', 'TransformPositionX', 'NUMERIC',
                '$.GameObjects[*].Transform.Position.Y', 'AS', 'TransformPositionY', 'NUMERIC',
                '$.GameObjects[*].Transform.Position.Z', 'AS', 'TransformPositionZ', 'NUMERIC',
                '$.GameObjects[*].Transform.Rotation.X', 'AS', 'TransformRotationX', 'NUMERIC',
                '$.GameObjects[*].Transform.Rotation.Y', 'AS', 'TransformRotationY', 'NUMERIC',
                '$.GameObjects[*].Transform.Rotation.Z', 'AS', 'TransformRotationZ', 'NUMERIC',
                '$.GameObjects[*].Transform.Scale.X', 'AS', 'TransformScaleX', 'NUMERIC',
                '$.GameObjects[*].Transform.Scale.Y', 'AS', 'TransformScaleY', 'NUMERIC',
                '$.GameObjects[*].Transform.Scale.Z', 'AS', 'TransformScaleZ', 'NUMERIC'
            ]
            
    redis_client.execute_command(*create_command_args)
            
    print(f"Successfully created index '{INDEX_NAME}'.")

Index 'GameObjectsIdx' not found. Creating index now...
Successfully created index 'GameObjectsIdx'.
